# Module 27 – Cardiac Diagnostics Case Study
**Goal:** Build a machine learning model that predicts the presence of heart disease from patient health data.

**Dataset:** Heart Disease UCI (Kaggle). Expected columns: `age, sex, cp, trestbps, chol, fbs, restecg, thalach, exang, oldpeak, slope, ca, thal, target`
(the 920-row version has `num` instead of `target`, plus `id` and `dataset`; both are handled below).

**Workflow:** 1. Import & explore → 2. Clean & preprocess → 3. Train/test split → 4. Models → 5. Evaluation → 6. Summary

## 1. Import libraries and load the data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, ConfusionMatrixDisplay, classification_report)

sns.set_theme(style='whitegrid')
RANDOM_STATE = 42

**Loading the data.** Download the CSV from Kaggle (search *"Heart Disease UCI"*), then either upload it in the Colab file panel
as `heart.csv`, or run the upload cell below. If no file is found, the notebook falls back to the original UCI repository.

In [ ]:
import os

CSV_PATH = 'heart.csv'   # change if your file has a different name

if not os.path.exists(CSV_PATH):
    try:
        from google.colab import files
        print('Upload the Kaggle heart disease CSV (or cancel to use the UCI fallback)...')
        up = files.upload()
        if up:
            CSV_PATH = list(up.keys())[0]
    except Exception:
        pass

if os.path.exists(CSV_PATH):
    df = pd.read_csv(CSV_PATH)
    print(f'Loaded {CSV_PATH}')
else:
    # Fallback: original UCI Heart Disease data (needs internet)
    !pip -q install ucimlrepo
    from ucimlrepo import fetch_ucirepo
    hd = fetch_ucirepo(id=45)
    df = hd.data.features.copy()
    df['target'] = hd.data.targets['num']
    print('Loaded UCI Heart Disease dataset (fallback)')

print(df.shape)
df.head()

## 2. Explore the dataset

In [ ]:
df.info()

In [ ]:
df.describe(include='all').T

**Feature guide:** `age` (years), `sex` (1 = male), `cp` (chest pain type), `trestbps` (resting blood pressure), `chol` (serum cholesterol mg/dl),
`fbs` (fasting blood sugar > 120), `restecg` (resting ECG), `thalach` (max heart rate), `exang` (exercise-induced angina),
`oldpeak` (ST depression), `slope` (ST slope), `ca` (major vessels coloured by fluoroscopy), `thal` (thalassemia), `target` (heart disease).

## 3. Data cleaning and preprocessing

In [ ]:
# --- Standardise column names / drop identifier columns ---
df.columns = [c.strip().lower() for c in df.columns]
if 'num' in df.columns and 'target' not in df.columns:
    df = df.rename(columns={'num': 'target'})
df = df.drop(columns=[c for c in ['id', 'dataset'] if c in df.columns])

# --- Make the target binary: 0 = no disease, 1 = disease ---
df['target'] = (pd.to_numeric(df['target'], errors='coerce') > 0).astype(int)

# --- Convert boolean-like text columns (some versions use True/False) ---
for c in ['fbs', 'exang']:
    if c in df.columns and df[c].dtype == object:
        df[c] = df[c].map({'True': 1, 'False': 0, True: 1, False: 0})

# --- In the UCI data, 0 cholesterol / blood pressure are impossible values => missing ---
for c in ['chol', 'trestbps']:
    if c in df.columns:
        n = (df[c] == 0).sum()
        if n: print(f'{c}: {n} impossible zero values set to NaN')
        df.loc[df[c] == 0, c] = np.nan

print('Target balance:\n', df['target'].value_counts(normalize=True).round(3))

In [ ]:
# --- Missing values ---
missing = df.isna().sum()
print(missing[missing > 0] if missing.any() else 'No missing values')

# --- Duplicates (duplicates across train/test would inflate accuracy) ---
dups = df.duplicated().sum()
print('Duplicate rows:', dups)
df = df.drop_duplicates().reset_index(drop=True)
print('Shape after removing duplicates:', df.shape)

### Exploratory visualisations

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
sns.countplot(x='target', data=df, ax=axes[0]); axes[0].set_title('Target distribution (1 = disease)')
sns.histplot(data=df, x='age', hue='target', kde=True, ax=axes[1]); axes[1].set_title('Age by target')
sns.boxplot(x='target', y='chol', data=df, ax=axes[2]); axes[2].set_title('Cholesterol by target')
plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(10, 7))
sns.heatmap(df.apply(pd.to_numeric, errors='coerce').corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation matrix'); plt.show()

### Encoding and scaling
- **Numeric** features (`age, trestbps, chol, thalach, oldpeak`): median imputation + standardisation (important for Logistic Regression and KNN).
- **Categorical** features (`sex, cp, fbs, restecg, exang, slope, ca, thal`): most-frequent imputation + one-hot encoding.

These steps live inside a `Pipeline`, so they are fitted **only on the training data** (no data leakage).

In [ ]:
numeric_features = [c for c in ['age', 'trestbps', 'chol', 'thalach', 'oldpeak'] if c in df.columns]
categorical_features = [c for c in df.columns if c not in numeric_features + ['target']]
print('Numeric    :', numeric_features)
print('Categorical:', categorical_features)

numeric_pipe = Pipeline([('imputer', SimpleImputer(strategy='median')),
                         ('scaler', StandardScaler())])
categorical_pipe = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')),
                             ('onehot', OneHotEncoder(handle_unknown='ignore'))])

preprocessor = ColumnTransformer([('num', numeric_pipe, numeric_features),
                                  ('cat', categorical_pipe, categorical_features)])

## 4. Train / test split

In [ ]:
X = df.drop(columns='target')
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

print('Train:', X_train.shape, ' Test:', X_test.shape)
print('Disease rate  train/test:', round(y_train.mean(), 3), '/', round(y_test.mean(), 3))

## 5. Train and compare classification models

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    'KNN':                 KNeighborsClassifier(n_neighbors=7),
    'Decision Tree':       DecisionTreeClassifier(max_depth=4, random_state=RANDOM_STATE),
    'Random Forest':       RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
fitted, rows = {}, []

for name, clf in models.items():
    pipe = Pipeline([('prep', preprocessor), ('model', clf)])
    cv_acc = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='accuracy').mean()
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    fitted[name] = pipe
    rows.append({'Model': name,
                 'CV Accuracy': cv_acc,
                 'Test Accuracy': accuracy_score(y_test, pred),
                 'Precision': precision_score(y_test, pred),
                 'Recall': recall_score(y_test, pred),
                 'F1-score': f1_score(y_test, pred)})

results = pd.DataFrame(rows).set_index('Model').round(3).sort_values('F1-score', ascending=False)
results

## 6. Detailed evaluation of the best model
Model selection uses **F1-score** (a balance of precision and recall). In a medical screening context, **recall** (catching actual
disease cases) is especially important, because a false negative means a missed diagnosis.

In [ ]:
best_name = results.index[0]
best_model = fitted[best_name]
y_pred = best_model.predict(X_test)
print('Best model:', best_name)

print(f'Accuracy : {accuracy_score(y_test, y_pred):.3f}')
print(f'Precision: {precision_score(y_test, y_pred):.3f}')
print(f'Recall   : {recall_score(y_test, y_pred):.3f}')
print(f'F1-score : {f1_score(y_test, y_pred):.3f}\n')
print(classification_report(y_test, y_pred, target_names=['No disease', 'Disease']))

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred),
                       display_labels=['No disease', 'Disease']).plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title(f'Confusion matrix – {best_name}')
plt.show()

tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
print(f'True negatives: {tn} | False positives: {fp} | False negatives (missed cases): {fn} | True positives: {tp}')

In [ ]:
# Compare all models visually
results[['Test Accuracy', 'Precision', 'Recall', 'F1-score']].plot(kind='bar', figsize=(10, 5), rot=15)
plt.title('Model comparison on the test set'); plt.ylim(0, 1.05); plt.legend(loc='lower right'); plt.show()

### Which features matter most?

In [ ]:
rf = fitted['Random Forest']
feat_names = rf.named_steps['prep'].get_feature_names_out()
imp = pd.Series(rf.named_steps['model'].feature_importances_, index=feat_names).sort_values(ascending=False).head(12)

plt.figure(figsize=(8, 5))
sns.barplot(x=imp.values, y=[n.split('__')[1] for n in imp.index], color='steelblue')
plt.title('Top 12 feature importances (Random Forest)'); plt.xlabel('Importance'); plt.show()

## 7. Summary of analysis and key findings

In [ ]:
b = results.loc[best_name]
print(f'''
SUMMARY
-------
Dataset : {df.shape[0]} patients (after removing duplicates), {df.shape[1]-1} features; {y.mean():.0%} have heart disease.
Pipeline: median/mode imputation, standardisation of numeric features, one-hot encoding of categorical features
          (all fitted on training data only), stratified 80/20 split, 5-fold CV.
Best model: {best_name}
  Accuracy {b['Test Accuracy']:.2f} | Precision {b['Precision']:.2f} | Recall {b['Recall']:.2f} | F1 {b['F1-score']:.2f}
  Missed disease cases (false negatives) on the test set: {fn}
Top predictors (Random Forest): {', '.join(n.split('__')[1] for n in imp.index[:5])}
''')

### Written conclusions
*(Edit these after running the notebook to match your own results.)*

1. **Data:** The dataset had a few missing/impossible values (e.g. zero cholesterol) that were treated as missing and imputed, and duplicate rows were removed to avoid train/test leakage. Classes are fairly balanced, so accuracy is a meaningful metric, but I also report precision, recall and F1.
2. **Models:** Four classifiers were compared (Logistic Regression, KNN, Decision Tree, Random Forest). The best performer by F1-score is shown in the results table above.
3. **Key predictors:** Chest pain type, maximum heart rate (`thalach`), ST depression (`oldpeak`), number of major vessels (`ca`) and thalassemia (`thal`) are typically among the strongest predictors, which matches clinical intuition.
4. **Clinical considerations:** False negatives (missed disease) are costlier than false positives, so in practice one might lower the decision threshold to raise recall, at the cost of more false alarms.
5. **Limitations & next steps:** The dataset is small, so results vary with the split. Next steps: hyper-parameter tuning (`GridSearchCV`), ROC-AUC analysis, and validation on an independent dataset. This model is for learning purposes and not a substitute for clinical diagnosis.